In [1]:
import pandas as pd
import os

In [2]:
path = r"C:\Users\APC\2526-LTXLDL-Project-1.5\src\processed\clean_data_monthly"

In [3]:
frames = []
for file in os.listdir(path):
    if file.endswith(".parquet"):
        file_path = os.path.join(path, file)
        temp = pd.read_parquet(file_path)
        frames.append(temp)
df = pd.concat(frames)

## Feature engine

In [4]:
df["pickup_date"] = df["tpep_pickup_datetime"].dt.date
df["pickup_hour"] = df["tpep_pickup_datetime"].dt.hour
df["pickup_dow"] = df["tpep_pickup_datetime"].dt.day_name().astype('category')
df["pickup_month"] = df["tpep_pickup_datetime"].dt.month
# df["pickup_week"] = df["tpep_pickup_datetime"].dt.to_period('W')

In [5]:
df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,trip_duration_minutes,speed_mph,pickup_date,pickup_hour,pickup_dow,pickup_month
0,2,2023-01-01 00:32:10,2023-01-01 00:40:36,1,0.97,1,N,161,141,2,...,1.0,14.30,2.5,0.0,8.433333,6.901137,2023-01-01,0,Sunday,1
1,2,2023-01-01 00:55:08,2023-01-01 01:01:27,1,1.10,1,N,43,237,1,...,1.0,16.90,2.5,0.0,6.316667,10.448450,2023-01-01,0,Sunday,1
2,2,2023-01-01 00:25:04,2023-01-01 00:37:49,1,2.51,1,N,48,238,1,...,1.0,34.90,2.5,0.0,12.750000,11.811709,2023-01-01,0,Sunday,1
3,2,2023-01-01 00:10:29,2023-01-01 00:21:19,1,1.43,1,N,107,79,1,...,1.0,19.68,2.5,0.0,10.833333,7.919956,2023-01-01,0,Sunday,1
4,2,2023-01-01 00:50:34,2023-01-01 01:02:52,1,1.84,1,N,161,137,1,...,1.0,27.80,2.5,0.0,12.300000,8.975566,2023-01-01,0,Sunday,1


## Tính KPI theo ngày

In [ ]:
kpi_daily = df.groupby('pickup_date').agg(
        total_trips=('tpep_pickup_datetime', 'size'), # Tính tổng số chuyến đi
        total_revenue=('total_amount', 'sum'),  # Tính tổng doanh thu
        total_distance_miles=('trip_distance', 'sum'),  # Tính tổng khoảng cách chuyến đi
        median_duration_minutes=('trip_duration_minutes', 'median'),  #Tính p50 thời gian đi
        p95_duration_minutes=('trip_duration_minutes', lambda x: x.quantile(0.95)),
        median_speed_mph=('speed_mph', 'median'), # p50 speed
        avg_passengers=('passenger_count', 'mean') # Trung bình số người đi
    )

avg_daily_trips = kpi_daily['total_trips'].mean() # Tính trung bình số chuyến đi trong 1 ngày
kpi_daily['index_trips_100'] = round((kpi_daily['total_trips'] / avg_daily_trips) * 100, 2) # công thức này sẽ cho biết tỉ lệ so với chuyến đi trung bình trong ngày (Index(100) theo ngày trong yêu cầu)

Lưu file

In [7]:
daily_path = f'processed/kpi_daily_2023.csv'
kpi_daily.to_csv(daily_path, encoding='utf-8-sig')
print(f"Đã lưu KPI theo ngày: {daily_path}")

Đã lưu KPI theo ngày: processed/kpi_daily_2023.csv


In [8]:
df['pickup_date']

0          2023-01-01
1          2023-01-01
2          2023-01-01
3          2023-01-01
4          2023-01-01
              ...    
2719196    2023-12-31
2719197    2023-12-31
2719198    2023-12-31
2719199    2023-12-31
2719200    2023-12-31
Name: pickup_date, Length: 30673803, dtype: object

#### KPI theo tuần

In [9]:
dow_day_counts = df.groupby('pickup_dow', observed=True)['pickup_date'].nunique() # Lọc ra những ngày khác biệt theo thứ

kpi_dow = df.groupby('pickup_dow', observed=True).agg(
        total_trips=('tpep_pickup_datetime', 'size'), # sum
        total_revenue=('total_amount', 'sum'), # sum
        median_duration_minutes=('trip_duration_minutes', 'median'), # p50 duration
        p95_duration_minutes=('trip_duration_minutes', lambda x: x.quantile(0.95)),  # p95(Phân vị 95%): 95% dữ liệu nhỏ hơn giá trị này
        median_speed_mph=('speed_mph', 'median')   # p50: Phân vị 50%, tức là median
)

# Chuẩn hóa để có số liệu trung bình/ngày vì dùng total_trip thì chủ nhật có 53 cái thì sẽ không phán ánh đúng
kpi_dow['avg_trips_per_day'] = kpi_dow['total_trips'] / dow_day_counts # Tổng số trip / số lượng ngày của thứ đó
kpi_dow['avg_revenue_per_day'] = kpi_dow['total_revenue'] / dow_day_counts

# Tương tự như kpi theo ngày (Indexdow theo thứ trong tuần)
avg_dow_trips_overall = kpi_dow['avg_trips_per_day'].mean()
kpi_dow['index_dow_trips'] = round((kpi_dow['avg_trips_per_day'] / avg_dow_trips_overall) * 100, 2)

# Sắp xếp lại thứ tự thứ 
days_of_week_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
kpi_dow = kpi_dow.reindex(days_of_week_order)

In [10]:
dow_day_counts

pickup_dow
Friday       52
Monday       52
Saturday     52
Sunday       53
Thursday     52
Tuesday      52
Wednesday    52
Name: pickup_date, dtype: int64

In [11]:
kpi_dow

,total_trips,total_revenue,median_duration_minutes,p95_duration_minutes,median_speed_mph,avg_trips_per_day,avg_revenue_per_day,index_dow_trips
pickup_dow,,,,,,,,
Monday,3737583,9.989148e+07,11.633333,39.200000,9.728414,71876.596154,1.920990e+06,85.50
Tuesday,4399704,1.154201e+08,12.550000,38.466667,8.825441,84609.692308,2.219617e+06,100.65
Wednesday,4648428,1.223957e+08,12.783333,39.150000,8.702250,89392.846154,2.353764e+06,106.34
Thursday,4738163,1.259142e+08,12.933333,40.233333,8.647306,91118.519231,2.421428e+06,108.39
Friday,4449017,1.169678e+08,12.300000,39.933333,9.048041,85558.019231,2.249380e+06,101.78
Saturday,4731939,1.228411e+08,12.100000,37.883333,9.805310,90998.826923,2.362328e+06,108.25
Sunday,3968969,1.080342e+08,11.466667,39.666667,10.914864,74886.207547,2.038381e+06,89.08


Lưu file

In [12]:
weekly_path = f'processed/kpi_weekly_2023.csv'
kpi_dow.to_csv(weekly_path, encoding='utf-8-sig')
print(f"Đã lưu KPI theo thứ trong tuần: {weekly_path}")

Đã lưu KPI theo thứ trong tuần: processed/kpi_weekly_2023.csv


### Tính KPI theo tháng

In [13]:
kpi_monthly = df.groupby('pickup_month').agg(
    total_trips=('tpep_pickup_datetime', 'size'), # sum
    total_revenue=('total_amount', 'sum'), # sum
    total_miles=('trip_distance', 'sum'), # Tổng quãng đường
    median_duration_minutes=('trip_duration_minutes', 'median'), # p50 duration
    p95_duration_minutes=('trip_duration_minutes', lambda x: x.quantile(0.95)),  # p95
    median_speed_mph=('speed_mph', 'median')
)   

In [14]:
kpi_monthly

,total_trips,total_revenue,total_miles,median_duration_minutes,p95_duration_minutes,median_speed_mph
pickup_month,,,,,,
1,2479184,62388105.41,7386083.30,11.216667,33.366667,9.969120
2,2356276,59209355.19,6914102.23,11.516667,34.000000,9.711547
3,2712746,70019067.52,8151036.20,11.833333,36.883333,9.565134
4,2650203,69849171.67,8268305.09,12.050000,38.750000,9.666927
5,2823933,76035683.58,8884799.12,12.616667,40.833333,9.291541
6,2639798,71130116.41,8326259.46,12.400000,40.450000,9.405595
7,2331205,61905870.82,7449424.57,12.016667,38.916667,9.795798
8,2247939,59680168.46,7221214.72,12.000000,39.433333,9.795597
9,2255288,62289053.36,7052079.59,13.116667,43.333333,8.783396


In [ ]:
# Tính % tổng năm theo tháng (% số chuyến đi, % doanh thu revenue)
total_annual_trips = kpi_monthly['total_trips'].sum()
kpi_monthly['percent_of_annual_trips'] = round((kpi_monthly['total_trips'] / total_annual_trips) * 100, 2)

total_annual_revenue = kpi_monthly['total_revenue'].sum()
kpi_monthly['percent_of_annual_revenue'] = round((kpi_monthly['total_revenue'] / total_annual_revenue) * 100, 2)

In [16]:
kpi_monthly

,total_trips,total_revenue,total_miles,median_duration_minutes,p95_duration_minutes,median_speed_mph,percent_of_annual_trips,percent_of_annual_revenue
pickup_month,,,,,,,,
1,2479184,62388105.41,7386083.30,11.216667,33.366667,9.969120,8.08,7.69
2,2356276,59209355.19,6914102.23,11.516667,34.000000,9.711547,7.68,7.30
3,2712746,70019067.52,8151036.20,11.833333,36.883333,9.565134,8.84,8.63
4,2650203,69849171.67,8268305.09,12.050000,38.750000,9.666927,8.64,8.61
5,2823933,76035683.58,8884799.12,12.616667,40.833333,9.291541,9.21,9.37
6,2639798,71130116.41,8326259.46,12.400000,40.450000,9.405595,8.61,8.77
7,2331205,61905870.82,7449424.57,12.016667,38.916667,9.795798,7.60,7.63
8,2247939,59680168.46,7221214.72,12.000000,39.433333,9.795597,7.33,7.35
9,2255288,62289053.36,7052079.59,13.116667,43.333333,8.783396,7.35,7.68


In [17]:
monthly_path = f'processed/kpi_monthly_2023.csv'
kpi_monthly.to_csv(monthly_path, encoding='utf-8-sig')
print(f"Đã lưu KPI theo thứ trong tuần: {monthly_path}")

Đã lưu KPI theo thứ trong tuần: processed/kpi_monthly_2023.csv
